# Gold Layer: Market Baselines Fact Notebook
Calculates price benchmarks (minimum, median, maximum) and active ad counts for each hardware product configuration (`baseline_id`).

## 1. Setup and Imports
Initialize environment path and import SQLAlchemy, Polars, and model layer modules.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, insert
from sqlalchemy.orm import Session
import polars as pl

# Database configuration and model layer namespaces
from app.config import db_engine
from app.models import silver, gold

## 2. Query Silver Layer Listings
Fetch active clean ads and hardware specification attributes required for baseline computation.

In [ ]:
with db_engine.connect() as connection:
    df_silver_raw = pl.read_database(
        select(
            silver.SilverCleanAd.category,
            silver.SilverCleanAd.cpu_brand,
            silver.SilverCleanAd.ram_gb,
            silver.SilverCleanAd.storage_gb,
            silver.SilverCleanAd.ad_id,
            silver.SilverCleanAd.price,
            silver.SilverCleanAd.date,
            silver.SilverCleanAd.baseline_id
        ),
        connection=connection
    )

## 3. Compute Baseline Aggregates
Filter for the latest listing date per `baseline_id` and compute pricing metrics (min, median, max) and active ad volumes.

In [ ]:
market_baselines_df = (
    df_silver_raw
    # Filter for latest listing date per baseline configuration
    .filter(
        pl.col('date') == pl.col('date').max().over('baseline_id')
    )
    .group_by(['baseline_id', 'category', 'cpu_brand', 'ram_gb', 'storage_gb'])
    .agg(
        pl.col('ad_id').count().alias('active_ads_count'),
        pl.col('price').min().alias('min_price'),
        pl.col('price').median().alias('median_price'),
        pl.col('price').max().alias('max_price'),
        pl.col('date').max().alias('last_recalculated_at'),
    )
)

## 4. Persist to Gold Market Baseline Fact Table (`ft_gold_market_baselines`)
Save computed baselines to `gold.FactMarketBaseline`.

In [ ]:
if not market_baselines_df.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(gold.FactMarketBaseline), market_baselines_df.to_dicts()
        )
        session.commit()
        print(f"Successfully saved {len(market_baselines_df)} baselines to ft_gold_market_baselines.")
else:
    print("No baseline data found to insert.")